# Finding and Analzying Securities for Supporting Investment Decisions for 2025

## Introduction
The aim of this project is to gather information from the internet about suggested stocks, and analyse them.
**Objectives**:
- Define short-term investmen
- Use a webscraper to collect information about ssuggested securities
- Store the collected data in a database
- Select the 3 most frequently mentioned securities
- Prepare data for analysis
- Analyse data
- Summarize insights

### What are the benefits of investing in growth stocks?
Growth stocks are shares of companies that are expected to grow significantly faster than the overall market or their industry peers. These companies typically reinvest their earnings into the business to fuel further expansion, which often leads to impressive revenue and profit growth over time.1

### What will the analysis be focus on?
The analysis focuses on:
- visualize the most frequent words used when mentioning the specific stocks
- analyze the context around the stock's company; is it mostly positive or negative? Probably we don't want to invest in a company which has a negative publicity
- *volatility*: check the difference between the highest and lowest values within a day. Also, examine the difference between the opening and closing prices on the same day, and compare the opening price with the previous day's closing price to see the mood changes while the market is closed.
- *monotonity*: analyze the closing prices to determine the overall trend of the security

# SQL Database
Ceate a database to store data which will be analysed later. 

Considerations:
- Rarely, but it can happen that one company has more tickers
- When looking for tickers, it may not be possible to scrape company name and sector from the texts at first, so those are optional, can be filled with later searches
- Context is text scraped from where the company and/or sector was mentioned if exists
- Link is the URL of the website where the ccompany and/or sector was mentioned if exists

#### Tables

**stock_info**  
Ticker - text, primary key  
Company name - optional text,  
Sector - optional text,
Investopedia_link - optional text
  
**company_context**  
Company - text, primary key  
Context - optional text,  
link - optional text  

**sector_context**  
Sector - text, primary key  
Context - optional text,  
Link - optional text  






## Create tables

In [1]:
import sqlite3 as sql

# To avoid conflict with different versions
%config SqlMagic.style = 'Default'
%reload_ext sql

In [2]:
%sql sqlite:///stockAnalysis.db DROP TABLE stock_info;
%sql sqlite:///stockAnalysis.db DROP TABLE company_context;
%sql sqlite:///stockAnalysis.db DROP TABLE sector_context;

Done.
Done.
Done.


[]

In [3]:
%%sql sqlite:///stockAnalysis.db
CREATE TABLE IF NOT EXISTS stock_info(
    ticker VARCHAR(10) NOT NULL UNIQUE,
    company TEXT,
    sector VARCHAR(100),
    investopedia_link TEXT,
    PRIMARY KEY (ticker),
    FOREIGN KEY (company) REFERENCES company_context(company),
    FOREIGN KEY (sector) REFERENCES sector_context(sector)
);

Done.


[]

In [4]:
%%sql sqlite:///stockAnalysis.db
CREATE TABLE IF NOT EXISTS company_context(
    company TEXT NOT NULL UNIQUE,
    title TEXT,
    article TEXT,
    link TEXT, 
    PRIMARY KEY (company)
);

Done.


[]

In [5]:
%%sql sqlite:///stockAnalysis.db
CREATE TABLE  IF NOT EXISTS sector_context(
    sector VARCHAR(100) NOT NULL UNIQUE,
    title TEXT, 
    article TEXT,
    link TEXT, 
    PRIMARY KEY (sector)
);

Done.


[]

##### Show created tables

In [6]:
%%sql sqlite:///stockAnalysis.db
SELECT name
FROM sqlite_master
WHERE type='table';

Done.


name
stock_info
company_context
sector_context


## Handle Database queries

In [7]:
from io import StringIO
from contextlib import redirect_stdout

STOCK_DB = "%sql sqlite:///stockAnalysis.db"
MOCK_DB = "%sql sqlite:///:memory:"

class StockAnalysisDB:
    debug = None
    mock_db = None

    def __init__(self, mock_db = False, debug = False):
        self.debug = debug
        self.mock_db = mock_db
        
    def validate_dict_raise(self, data):
        if not (isinstance(data, list) and all(isinstance(row, dict) for row in data)):
            raise TypeError("Data is not a list of dictionaries")

    def call_query(self, query):
        # sql magic doesn't throw just writes to stdout
        ret = None
        output = StringIO()
        with redirect_stdout(output):
            if self.mock_db:
                if self.debug:
                    print("habzsi mock db is used")
                ret = %sql sqlite:///:memory: {query}
            else:
                if self.debug:
                    print("habzsi stock db is used")
                ret = %sql sqlite:///stockAnalysis.db {query}
        
        msg = output.getvalue()
        if self.debug:
            print(msg)
        
        if 'sqlite3.OperationalError' in msg:
            raise sql.OperationalError(msg)

        if 'sqlite3.IntegrityError' in msg:
            raise sql.IntegrityError(msg)

        return ret
    
    def insert_into(self, table, data):
        ## Make an SQL-injection check for table and data!!
        try:
            # data must be a list of dictionaries
            self.validate_dict_raise(data)

            queries = []
            for row in data:
                columns = ", ".join(row.keys())    
                values = ", ".join(
                    f"'{v}'" if isinstance(v, str) and v!= "N/A" else "NULL"
                    for v in row.values()
                )
                query = f"INSERT INTO {table} ({columns}) VALUES({values});" 
                self.call_query(query)
        
        except TypeError as e:
            print(e)
        except sql.OperationalError as e:
            print(f"SQL insert into Failed due to operation error: {e}")
            raise

        except sql.IntegrityError as e:
            print(f"SQL insert into failed due to integrity error: {e}")
            raise

    def select(self, table, columns = None, where = None):
        try:
            query = "SELECT "
            
            if columns:
                query +=", ".join(columns)
            else:
                query += "*"
            
            query +=" FROM " + table

            if where:
                query +=" WHERE " + where;
            
            query += ";"
            return self.call_query(query)
            
        except TypeError as e:
            print(e)
        except sql.OperationalError as e:
            print(f"SQL Select Failed due to operation error: {e}")
            raise
        except sql.IntegrityError as e:
            print(f"SQL Select failed due to integrity error: {e}")
            raise

# Web Spider
I will gather data from the Internet using a Webscraper:
- Tickers mentioned as good investment oportunities, whenever possible I will also save the company and sector
- I will save context from atricles about the company, and the sector

#### Ethical Considerations
...

## Webscraper
- Gather data about AI stocks from a website
- Identify the tickers in the artices
- Store data gathered in a database


In [48]:
import re
from urllib.request import urlopen
from urllib.request import Request
from urllib.error import HTTPError, URLError
# from urllib.parse import urljoin
from bs4 import BeautifulSoup
import time

def get_bs_from_url(url):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
            "Referer": "https://www.google.com/", # simulate Google search
            "Connection": "keep-alive",
            "Upgrade-Insecure-Requests": "1"
        }
        
        # To avoid 'Too many requests'
        time.sleep(5)
        
        response = urlopen(Request(url, headers=headers))
        return BeautifulSoup(response.read(), 'html.parser')

    except HTTPError as e:
        print(f"HTTP error: {e.code} - {e.reason}")
        return None
    
    except URLError as e:
        print(f"URL error for {url}: {e.reason}")
        return None

    except ValueError  as e:
        print(f"ValueError for {url}: {e.reason}")

BASE_URL = "https://www.investopedia.com/the-best-growth-stocks-8783031"
class InvestopediaWebSpider:
    stock_db = None
    data_cleaner = None
    debug = None
    bs = None
    links_to_explore = set()
    visited_links = set()
    links_with_context_map = {}

    def __init__(self, mock_db = False, debug = False):
        self.stock_db = StockAnalysisDB(mock_db=mock_db, debug=debug)
        self.data_handler = StockDataHandler(mock_db=mock_db, debug=debug)
        self.debug = debug
        self.bs = get_bs_from_url(BASE_URL)
        if not self.bs:
            raise ValueError("BeautifulSoup is not existing, the class can't be initiatied")
    
    def scrape_ticker_data_from_investopedia(self):
        # Look for the table with the world 'stock'
        table = None
        for tbl in self.bs.find_all("table"):
            if tbl.find("th", string=lambda s: s and "stock" in s.lower()):
                table = tbl
                break

        if table:
            # First row of the table body contains the indexes
            first_row = table.find("tbody").find("tr")
            column_name_indexes = {}
            for i, td in enumerate(first_row.find_all("td", recurisive=False)):
                text = td.get_text(strip=True).lower()
                if "ticker" in text:
                    column_name_indexes["ticker"] = i
                elif "company" in text:
                    column_name_indexes["company"] = i
                elif "sector" in text:
                    column_name_indexes["sector"] = i
                i+=1

            # Get next data from table
            all_ticker_data = []
            for row in first_row.find_next_siblings("tr"):
                ticker_data = {}
                cells = row.find_all("td")
                
                # Ticker has a link to the profile
                link = cells[column_name_indexes["ticker"]].find('a')
                if link:
                    ticker_data["investopedia_link"] = link.attrs['href']
                    self.links_to_explore.add(link.attrs['href'])
                
                # Extract data from each cell
                ticker_data["ticker"] = cells[column_name_indexes["ticker"]].get_text(strip=True)
                ticker_data["company"] = cells[column_name_indexes["company"]].get_text(strip=True)
                ticker_data["sector"] = cells[column_name_indexes["sector"]].get_text(strip=True)

                all_ticker_data.append(ticker_data)

            self.stock_db.insert_into("stock_info", all_ticker_data)

    # Start on te provided websites
        # Collect all the links without thinking to have tons of links we can filter
        # Go to links, and collect links there too
        # depth limit = 5
        # This will take a while, so print the statuses

    def recursive_collect_link_with_spider(self, base_link, max_depth = 3):

        # Ha useful link, lementsem optimalizalasbol?
        if max_depth == 0 or base_link in self.visited_links:
            return
        
        bs = get_bs_from_url(base_link)
        if not bs:
            return
            
        self.visited_links.add(base_link)
        
        for link in bs.find_all('a'):
            if 'href' in link.attrs:
                # Remove references we can't use
                if not link.attrs.get('href', '').startswith(('#', '/')) \
                    and 'javascript' not in link.attrs.get('href', '').lower():
                    
                    link_text = link.attrs['href']
                    # Go to next link if already visited
                    if link_text in self.visited_links:
                        continue

                    kw = self.data_handler.get_keyword_if_link_contains(link_text)
                    if kw:
                        self.links_with_context_map[kw] = link_text
                        print(f"Context found for {kw}")

                    print(f"Explore link:{link_text}")
                    self.recursive_collect_link_with_spider(link_text, max_depth-1)

    def gather_useful_links(self):
        for news_link in self.data_handler.NEWS_LINKS:
            print(f"Collecting URLs from {news_link}...")
            self.recursive_collect_link_with_spider(news_link)
        
        # keywords = self.data_cleaner.get_data_as_keyword_from_db(['ticker', 'company', 'sector']) \
        #          | self.data_cleaner.SEARCH_KEYWORDS
        # for links in self.links_to_explore:
        #     if any(keyword in link_text for keyword in (keywords)):
        #         self.links_to_explore.add(link_text)

    """
    Scrapes the collected URLs for any occurrence of tickers, companies and sector.
    If a keyword is found in a header, it also collects the next paragraph.
    Stores the context in the database.
    """
    def scrape_context_for_keywords(self): 
        pass
        #keyword_dict = {keyword: [] for keyword in data_as_keyword}
        
        # limit = 0
        # MAX_LIMIT = 40
        # for link in self.links_to_explore:
        #     if limit == MAX_LIMIT:
        #         break
            
        #     bs = get_bs_from_url(link)
        #     if bs is None:
        #         continue
            
        #     elements = bs.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p'])
        #     for i, e in enumerate(elements):
        #         text = e.get_text(separator=" ", strip=True)
        #         for keyword in data_as_keyword:
        #             context = []
                    
        #             # test with regex if it's a match (important for tickers as they are very small)
        #             if re.search(r'\b' + re.escape(keyword) + r'\b', text, re.IGNORECASE):    
        #                 text_from_siblings_and_under = ""
                       
        #                 # Gather all texts from siblings and under
        #                 for sibling in e.find_next_siblings():
        #                     text_from_siblings_and_under += sibling.get_text(separator=" ", strip=True)
                        
        #                 context.append(text_from_siblings_and_under)
        #                 keyword_dict[keyword].append(text + text_from_siblings_and_under)
            
        #     print(f"{limit}/{MAX_LIMIT} Link explored: " + link)
        #     limit += 1
        # return keyword_dict
            
        

Create spider class, and scrape investopedia for ticker data

In [49]:
spider = InvestopediaWebSpider()

In [50]:
# spider.scrape_ticker_data_from_investopedia()

In [51]:
spider.gather_useful_links()

HTTP error: 401 - HTTP Forbidden
HTTP error: 401 - HTTP Forbidden
Explore link:https://www.livemint.com/
Explore link:https://www.livemint.com/market/market-stats
Explore link:https://www.livemint.com/notifications
Explore link:https://epaper.livemint.com/?utm_source=LM_Desk_Header&utm_medium=LM_desk_top_nav&utm_campaign=Ref
Explore link:https://epaper.livemint.com/?utm_source=LM_Desk_Header&utm_medium=LM_desk_top_nav&utm_campaign=Ref
Explore link:https://rzp.io/rzp/rBmxqHo
Explore link:https://www.livemint.com/latest-news
Explore link:https://www.livemint.com/market
Explore link:https://www.livemint.com/news
Explore link:https://www.livemint.com/premium
Explore link:https://www.livemint.com/companies
Explore link:https://www.livemint.com/money
Explore link:https://www.livemint.com/sports/cricket-news/icc-champions-trophy
Explore link:https://www.livemint.com/money/loan/personal-loan?cta_position
Explore link:https://www.livemint.com/technology
Explore link:https://www.livemint.com/hin

AttributeError: 'ValueError' object has no attribute 'reason'

In [52]:
spider.links_with_context_map

{'RUA': 'https://www.livemint.com/market/stock-market-news/stocks-to-buy-today-stock-recommendations-today-marketsmith-india-s-top-stock-picks-for-28-february-market-outlook-nifty-11740659712077.html ',
 'INOD': 'https://api.whatsapp.com/send?text=FMCG, Banking among other sectors poised to gain on fair valuations: Expert view - https://www.livemint.com/market/stock-market-news/fmcg-banking-chemicals-among-other-sectors-poised-to-gain-on-fair-valuations-says-geojit-fin-services-vinod-nair-11740904344331.html?utm_source%3Dshare%26utm_medium%3Dsocial%26utm_campaign%3Dshare_via_web %0aSubscribe to Mint Premium 1 year plan using this link at ₹500 off - https://www.read.ht/Sb5N %0aDownload mint app for latest in Business News - https://bit.ly/32XEfFE'}

In [ ]:
# spider.links_to_explore

In [54]:
%sql sqlite:///stockAnalysis.db "SELECT * FROM stock_info;"

Done.


ticker,company,sector,investopedia_link
GRRR,Gorilla Technology Group Inc.,Technology Services,https://www.investopedia.com/markets/quote?tvwidgetsymbol=GRRR
ORGO,Organogenesis Holdings Inc.,Health Technology,https://www.investopedia.com/markets/quote?tvwidgetsymbol=ORGO
CYD,China Yuchai International Limited,Producer Manufacturing,https://www.investopedia.com/markets/quote?tvwidgetsymbol=CYD
SLDB,Solid Biosciences Inc.,Health Technology,https://www.investopedia.com/markets/quote?tvwidgetsymbol=SLDB
GDS,GDS Holdings Limited,Technology Services,https://www.investopedia.com/markets/quote?tvwidgetsymbol=GDS
PRCH,"Porch Group, Inc.",Technology Services,https://www.investopedia.com/markets/quote?tvwidgetsymbol=PRCH
ROOT,"Root, Inc.",Finance,https://www.investopedia.com/markets/quote?tvwidgetsymbol=ROOT
SWTX,"SpringWorks Therapeutics, Inc.",Health Technology,https://www.investopedia.com/markets/quote?tvwidgetsymbol=SWTX
VNET,"VNET Group, Inc.",Technology Services,https://www.investopedia.com/markets/quote?tvwidgetsymbol=VNET
INOD,Innodata Inc.,Technology Services,https://www.investopedia.com/markets/quote?tvwidgetsymbol=INOD


In [276]:
%sql sqlite:///stockAnalysis.db "DELETE FROM stock_info;"

11 rows affected.


[]

In [69]:
spider.collect_links_from_self_bs()

Number of links collected:247


In [213]:
# spider.links_to_explore

Spider through the links to gather context where the company or sector was mentioned

In [42]:
%sql sqlite:///stockAnalysis.db "SELECT * FROM company_context;"

Done.


company,title,article,link


In [13]:
%sql sqlite:///stockAnalysis.db "SELECT * FROM sector_context;"

Done.


sector,context,link


# Data Cleaning & Transformation
- Fetches raw data from DB
- Cleans missing values, standardizes formats
- Prepare for analysis: filtering, removing duplicates

In [10]:
import pandas as pd

class StockDataHandler():
    SEARCH_KEYWORDS={'company', 'sector', 'stock', 'news', 'analysis', 'report', 'press', 'overview', 'financial', 
                     'guidance', 'result', 'outlook', 'market', 'trend', 'corporate', 'strategy', 'research'}
    
    NEWS_LINKS={"https://www.investing.com/", "https://www.marketwatch.com/", "https://www.cnbc.com/world/?region=world",
                "https://www.livemint.com/market/stock-market-news/", "https://www.ft.com/markets", "https://www.nasdaq.com/",
                "https://edition.cnn.com/markets","https://finance.yahoo.com/topic/stock-market-news/", "https://www.reuters.com/markets/"}
    
    stock_db = None
    debug = None
    def __init__(self, mock_db = False, debug = False):
        self.stock_db = StockAnalysisDB(mock_db=mock_db, debug=debug)
        self.debug = debug
        self.__ticker_keywords = set()
    
    # Remove not-needed legal endings
    def simplify_company_names(self, data_set):
        clean_data = set()
        for company in data_set:
            cleaned_company = company.replace(" Group Inc.", "").replace(" Group, Inc.", "")\
                                     .replace(" Holdings Inc.", "").replace(" Holdings, Inc.", "")\
                                     .replace(" Holdings Limited", "").replace(" International Limited", "")\
                                     .replace(" Holdings LLC", "").replace(" International Limited", "")\
                                     .replace(" S.A.", "").replace(" Corp.", "").replace(" Co.", "")\
                                     .replace(", Inc.", "").replace(" Inc.", "").strip()
            clean_data.add(cleaned_company)
        return clean_data

    def get_data_as_keyword_from_db(self, columns=['ticker', 'company', 'sector']):
        
        data_df = self.stock_db.select("stock_info", columns).DataFrame()
        data_df = data_df.fillna("")

        # Sectors are too general, adding the text 'sector' will help with contex
        if 'sector' in data_df.columns:
            data_df['sector'] = data_df['sector'].apply(lambda s: s+ " sector")
        
        flattened_df = set(data_df.values.flatten())
        flattened_df.discard(" sector")                   
        flattened_df.discard("")
        return self.simplify_company_names(flattened_df)

    # At this state of the project the tickers won't change, so we can do this boosting
    def get_ticker_keywords(self):
        if len(self.__ticker_keywords) == 0:
            self.__ticker_keywords = self.get_data_as_keyword_from_db()
        return self.__ticker_keywords

    # Return keyword otherwise None
    def get_keyword_if_link_contains(self, link):
        if not isinstance(link, str):
            return None

        for kw in self.get_ticker_keywords():
            if kw.lower() in link.lower():
                return kw
        return None        

In [85]:
datah = StockDataHandler()
datah.get_keyword_if_link_contains("alma-korte-oro")

## Data preparation
First, I analyze the data and check the following:
- Validate if the expected columns are present
- Validate data formats and their defined domains
- Handle any missing data
- Fill NaN values using a custom strategy
- inspect zero values if any

# Optional: DataPipeline
If I have time and want to automate

# Data Analysis
- uses cleaned data from DB
- applies statistics, visualization, ML
- final output: insights

# Unit Tests

### Create mock tables in memory database

In [17]:
# To avoid conflict with different versions
%config SqlMagic.style = 'Default'
%reload_ext sql

In [18]:
%%sql sqlite:///:memory:
CREATE TABLE IF NOT EXISTS stock_info(
    ticker VARCHAR(10) NOT NULL UNIQUE,
    company TEXT,
    sector VARCHAR(100),
    investopedia_link TEXT,
    PRIMARY KEY (ticker),
    FOREIGN KEY (company) REFERENCES company_context(company),
    FOREIGN KEY (sector) REFERENCES sector_context(sector)
);

Done.


[]

In [19]:
%%sql sqlite:///:memory:
CREATE TABLE IF NOT EXISTS company_context(
    company TEXT NOT NULL UNIQUE,
    context TEXT,  
    link TEXT, 
    PRIMARY KEY (company)
);

Done.


[]

In [20]:
%%sql sqlite:///:memory:
CREATE TABLE  IF NOT EXISTS sector_context(
    sector VARCHAR(100) NOT NULL UNIQUE,
    context TEXT,  
    link TEXT, 
    PRIMARY KEY (sector)
);

Done.


[]

## Test Database class

In [171]:
%sql sqlite:///:memory:

import unittest
from io import StringIO
from unittest.mock import patch
import sqlite3 as sql

class TestDatabase(unittest.TestCase):

    def setUp(self):
        # Suppress stdout automatically
        self.patcher = patch('sys.stdout', new_callable=StringIO)
        self.mock_stdout = self.patcher.start()

    def tearDown(self):
        # Restore stdout after test
        self.patcher.stop()

    # Define test type when we expect no exception
    def assertNotRaises(self, exception, func, *args, **kwargs):
        try:
            func(*args, **kwargs)
        except exception as e:
            self.fail(f"Expected no {exception}, but got: {e}")
    
    def test_db_class_exists(self):
        self.assertIsNotNone(StockAnalysisDB)

    # Test insert data validator class

    def test_validate_dict_raise_exists(self):
        self.assertIsNotNone(StockAnalysisDB.validate_dict_raise)

    def test_validate_dict_raise_exception_empty_1(self):
        test_data = None
        with self.assertRaises(TypeError):
            StockAnalysisDB.validate_dict_raise(test_data)

    def test_validate_dict_raise_exception_empty_2(self):
        test_data = ""
        with self.assertRaises(TypeError):
            StockAnalysisDB.validate_dict_raise(test_data)

    def test_validate_dict_raise_exception_empty_3(self):
        test_data = []
        with self.assertRaises(TypeError):
            StockAnalysisDB.validate_dict_raise(test_data)

    def test_validate_dict_raise_exception_empty_4(self):
        test_data = {}
        with self.assertRaises(TypeError):
            StockAnalysisDB.validate_dict_raise(test_data)

    def test_validate_dict_raise_exception_not_list(self):
        test_dict = {'col1': 'link',
                        'col2': 'XXX',
                        'col3': 'text',
                        'col4': 'text'}
        with self.assertRaises(TypeError):
             StockAnalysisDB.validate_dict_raise(test_dict)

    def test_validate_dict_raise_exception_not_dict_in_list(self):
        test_list = ['text1', 'text2', 'text3', 'text4']
        with self.assertRaises(TypeError):
            StockAnalysisDB.validate_dict_raise(test_list)


    def test_validate_dict_not_raise_exception(self):
        my_test_list_dict = [{'col1': 'text', 'col2': 'XXX', 'col3': 'text', 'col4': 'text'},
                        {'col1': 'text', 'col2': 'RUA','col3': 'text', 'col4': 'text'}]
        db_instance = StockAnalysisDB(mock_db=True)
        self.assertNotRaises(TypeError, db_instance.validate_dict_raise, my_test_list_dict)

    # Test insert_into function
    
    def test_insert_into_exists(self):
        self.assertIsNotNone(StockAnalysisDB.insert_into)

    def test_insert_into_wrong_table(self):
        my_test_list_dict = [{'col1': 'text', 'col2': 'XXX', 'col3': 'text', 'col4': 'text'},
                        {'col1': 'text', 'col2': 'RUA','col3': 'text', 'col4': 'text'}]
        db_instance = StockAnalysisDB(mock_db=True)
        with self.assertRaisesRegex(sql.OperationalError, r"syntax error"):
            db_instance.insert_into("table", my_test_list_dict)

    def test_insert_into_wrong_column_name(self):
        my_test_list_dict = [{'col1': 'text', 'col2': 'XXX', 'col3': 'text', 'col4': 'text'},
                        {'col1': 'text', 'col2': 'RUA','col3': 'text', 'col4': 'text'}]
        db_instance = StockAnalysisDB(mock_db=True)
        with self.assertRaisesRegex(sql.OperationalError, r"has no column named"):
            db_instance.insert_into("stock_info", my_test_list_dict)

    def test_insert_into_no_primary_key(self):
        my_test_list_dict = [{'investopedia_link': 'https://www.investopedia.com/markets/quote?tvwidgetsymbol=TEM',
                              'company': 'Tempus AI Inc.',
                              'sector': 'Health Care'},]
        db_instance = StockAnalysisDB(mock_db=True)
        with self.assertRaisesRegex(sql.IntegrityError, r"constraint failed"):
            db_instance.insert_into("stock_info", my_test_list_dict)

    def test_insert_into_wrong_key_type(self):
        my_test_list_dict = [{'investopedia_link': 'https://www.investopedia.com/markets/quote?tvwidgetsymbol=TEM',
                              'ticker': 3,
                              'company': 'Tempus AI Inc.',
                              'sector': 'Health Care'},]
        db_instance = StockAnalysisDB(mock_db=True)
        with self.assertRaisesRegex(sql.IntegrityError, r"constraint failed"):
            db_instance.insert_into("stock_info", my_test_list_dict)

    def test_insert_into_correct(self):
        my_test_list_dict = [{'investopedia_link': 'link',
                              'ticker': 'XXX',
                              'company': 'company',
                              'sector': 'sector'}]
        db_instance = StockAnalysisDB(mock_db=True)
        db_instance.insert_into("stock_info", my_test_list_dict)

        res = %sql "SELECT investopedia_link,ticker,company,sector FROM stock_info WHERE ticker = 'XXX'"

        # Verify and delete data
        self.assertEqual(res[0][0], 'link')
        self.assertEqual(res[0][1], 'XXX')
        self.assertEqual(res[0][2], 'company')
        self.assertEqual(res[0][3], 'sector')

        %sql "DELETE FROM stock_info WHERE ticker='XXX'"

    # Test select function
    
    def test_select_exists(self):
        self.assertIsNotNone(StockAnalysisDB.select)

    @patch("__main__.StockAnalysisDB.call_query")
    def test_select_query_with_no_args(self, mock_execute):
        db_instance = StockAnalysisDB(mock_db=True)
        db_instance.select("table")

        # Check if query sting is correct
        mock_execute.assert_called_once_with("SELECT * FROM table;")

    @patch("__main__.StockAnalysisDB.call_query")
    def test_select_query_with_columns(self, mock_execute):
        db_instance = StockAnalysisDB(mock_db=True)
        db_instance.select("table", ['col1', 'col2', 'col3'])
    
        # Check if query sting is correct
        mock_execute.assert_called_once_with("SELECT col1, col2, col3 FROM table;")

    def test_select_return_with_no_args(self):
        db_instance = StockAnalysisDB(mock_db=True, debug=True)

        # Fill the mock db
        my_test_list_dict = [{'ticker': 'XXX',
                              'company': 'company',
                              'sector': 'sector',
                              'investopedia_link': 'link'}]
        db_instance.insert_into("stock_info", my_test_list_dict)

        # Verify and delete data
        res = db_instance.select("stock_info")
        self.assertEqual(len(res), 1)
        self.assertEqual(len(res[0]), 4)
        
        self.assertEqual(res[0][0], 'XXX')
        self.assertEqual(res[0][1], 'company')
        self.assertEqual(res[0][2], 'sector')
        self.assertEqual(res[0][3], 'link')

        %sql "DELETE FROM stock_info WHERE ticker='XXX'"

        def test_select_return_with_col(self):
            db_instance = StockAnalysisDB(mock_db=True, debug=True)
    
            # Fill the mock db
            my_test_list_dict = [{'ticker': 'XXX',
                                  'company': 'company',
                                  'sector': 'sector',
                                  'investopedia_link': 'link'},
                                  {'ticker': 'YYY',
                                  'company': 'company',
                                  'sector': 'sector',
                                  'investopedia_link': 'link'}]
            db_instance.insert_into("stock_info", my_test_list_dict)
    
            # Verify and delete data
            res = db_instance.select("stock_info", ['ticker'])
            self.assertEqual(len(res), 2)
            self.assertEqual(len(res[0]), 1)
            
            self.assertEqual(res[0][0], 'XXX')
            self.assertEqual(res[1][0], 'YYY')
    
            %sql "DELETE FROM stock_info WHERE ticker IN ('XXX', 'YYY')"

            def test_select_return_with_cols(self):
                db_instance = StockAnalysisDB(mock_db=True, debug=True)
        
                # Fill the mock db
                my_test_list_dict = [{'ticker': 'XXX',
                                      'company': 'company',
                                      'sector': 'sector',
                                      'investopedia_link': 'link'},
                                      {'ticker': 'YYY',
                                      'company': 'company2',
                                      'sector': 'sector',
                                      'investopedia_link': 'link'}]
                db_instance.insert_into("stock_info", my_test_list_dict)
        
                # Verify and delete data
                res = db_instance.select("stock_info", ['ticker', 'company'])
                self.assertEqual(len(res), 2)
                self.assertEqual(len(res[0]), 2)
                
                self.assertEqual(res[0][0], 'XXX')
                self.assertEqual(res[0][1], 'company')

                self.assertEqual(res[1][0], 'YYY')
                self.assertEqual(res[1][1], 'company2')
        
                %sql "DELETE FROM stock_info WHERE ticker IN ('XXX', 'YYY')"

suite = unittest.TestLoader().loadTestsFromTestCase(TestDatabase)
unittest.TextTestRunner(verbosity=2).run(suite)

test_db_class_exists (__main__.TestDatabase.test_db_class_exists) ... ok
test_insert_into_correct (__main__.TestDatabase.test_insert_into_correct) ... ok
test_insert_into_exists (__main__.TestDatabase.test_insert_into_exists) ... ok
test_insert_into_no_primary_key (__main__.TestDatabase.test_insert_into_no_primary_key) ... ok
test_insert_into_wrong_column_name (__main__.TestDatabase.test_insert_into_wrong_column_name) ... ok
test_insert_into_wrong_key_type (__main__.TestDatabase.test_insert_into_wrong_key_type) ... ok
test_insert_into_wrong_table (__main__.TestDatabase.test_insert_into_wrong_table) ... ok
test_select_exists (__main__.TestDatabase.test_select_exists) ... ok
test_select_query_with_columns (__main__.TestDatabase.test_select_query_with_columns) ... ok
test_select_query_with_no_args (__main__.TestDatabase.test_select_query_with_no_args) ... ok
test_select_return_with_no_args (__main__.TestDatabase.test_select_return_with_no_args) ... ok
test_validate_dict_not_raise_exceptio

<unittest.runner.TextTestResult run=19 errors=0 failures=0>

In [148]:
# %sql sqlite:///stockAnalysis.db "SELECT * FROM stock_info;"
res = %sql sqlite:///:memory: "SELECT * FROM stock_info;"

Done.


In [160]:
my_test_list_dict = [{'ticker': 'XXX',
                                  'company': 'company',
                                  'sector': 'sector',
                                  'investopedia_link': 'link'},
                                  {'ticker': 'YYY',
                                  'company': 'company',
                                  'sector': 'sector',
                                  'investopedia_link': 'link'}]
db_instance.insert_into("stock_info", my_test_list_dict)
    
            # Verify and delete data
res = db_instance.select("stock_info", ['ticker'])

In [199]:
# %sql sqlite:///:memory: "DELETE FROM stock_info;"

## Test BeautifulSoup function

In [89]:
%sql sqlite:///:memory:

class TestGetBSFromURL(unittest.TestCase):

    def setUp(self):
        # Suppress stdout automatically
        self.patcher = patch('sys.stdout', new_callable=StringIO)
        self.mock_stdout = self.patcher.start()

    def tearDown(self):
        # Restore stdout after test
        self.patcher.stop()
    
    # test what happens when bs can't be created by mocking the urlopen function
    @patch("__main__.urlopen")
    def test_http_error(self, mock_urlopen):
        mock_urlopen.side_effect = HTTPError(
            url = BASE_URL,
            code = 404,
            msg = 'Not Found',
            hdrs = {},
            fp = None)

        bs = get_bs_from_url(BASE_URL)
        self.assertIsNone(bs)

    def test_valid_url(self):
        bs = get_bs_from_url(BASE_URL)
        self.assertIsNotNone(bs)
        self.assertIn("Growth Stocks", str(bs))
    
    # test what happens when the link is wrong by mocking the urlopen function
    @patch("__main__.urlopen")
    def test_wrong_url_and_no_bs(self, mock_urlopen):
        mock_urlopen.side_effect = URLError("No route to host")
        bs = get_bs_from_url("https://www.invesftopedia.com/the-best-growth-stocks-8783031")
        self.assertIsNone(bs)

suite = unittest.TestLoader().loadTestsFromTestCase(TestGetBSFromURL)
unittest.TextTestRunner(verbosity=2).run(suite)

test_http_error (__main__.TestGetBSFromURL.test_http_error) ... ok
test_valid_url (__main__.TestGetBSFromURL.test_valid_url) ... ok
test_wrong_url_and_no_bs (__main__.TestGetBSFromURL.test_wrong_url_and_no_bs) ... ok

----------------------------------------------------------------------
Ran 3 tests in 15.392s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

## Test Web Spider class

In [88]:
class TestWebScraper(unittest.TestCase):
    def setUp(self):
        # Suppress stdout automatically
        self.patcher = patch('sys.stdout', new_callable=StringIO)
        self.mock_stdout = self.patcher.start()

    def tearDown(self):
        # Restore stdout after test
        self.patcher.stop()
    
    def test_webspider_class_exists(self):
        self.assertIsNotNone(InvestopediaWebSpider)

    # test scrape_ticker_data_from_investopedia function
    
    def test_scrape_ticker_data_exists(self):
        self.assertIsNotNone(InvestopediaWebSpider.scrape_ticker_data_from_investopedia)

    def test_scrape_ticker_data_fills_DB(self):
        test_spider = InvestopediaWebSpider(mock_db=True)
        test_spider.scrape_ticker_data_from_investopedia()

        # DB should be filled with expected results
        res = %sql "SELECT * FROM stock_info"
        expected_list = [('GRRR','Gorilla Technology Group Inc.', 'Technology Services','https://www.investopedia.com/markets/quote?tvwidgetsymbol=GRRR'),
        ('ORGO','Organogenesis Holdings Inc.', 'Health Technology','https://www.investopedia.com/markets/quote?tvwidgetsymbol=ORGO'),
        ('CYD','China Yuchai International Limited','Producer Manufacturing','https://www.investopedia.com/markets/quote?tvwidgetsymbol=CYD'),
        ('SLDB','Solid Biosciences Inc.','Health Technology', 'https://www.investopedia.com/markets/quote?tvwidgetsymbol=SLDB'),
        ('GDS', 'GDS Holdings Limited', 'Technology Services','https://www.investopedia.com/markets/quote?tvwidgetsymbol=GDS'),
        ('PRCH','Porch Group, Inc.','Technology Services','https://www.investopedia.com/markets/quote?tvwidgetsymbol=PRCH'),
        ('ROOT','Root, Inc.','Finance','https://www.investopedia.com/markets/quote?tvwidgetsymbol=ROOT'),
        ('SWTX','SpringWorks Therapeutics, Inc.','Health Technology','https://www.investopedia.com/markets/quote?tvwidgetsymbol=SWTX'),
        ('VNET','VNET Group, Inc.','Technology Services', 'https://www.investopedia.com/markets/quote?tvwidgetsymbol=VNET'),
        ('INOD','Innodata Inc.','Technology Services','https://www.investopedia.com/markets/quote?tvwidgetsymbol=INOD'),
        ('RUA',	'Russell 3000',	None,'https://www.investopedia.com/markets/quote?tvwidgetsymbol=RUA')]

        %sql "DELETE FROM stock_info;"

        self.assertListEqual(res, expected_list)

        # test collect_links_from_self_bs function
        def test_collect_links_exists(self):
            self.assertIsNotNone(InvestopediaWebSpider.collect_links_from_self_bs)

suite = unittest.TestLoader().loadTestsFromTestCase(TestWebScraper)
unittest.TextTestRunner(verbosity=2).run(suite)

test_scrape_ticker_data_exists (__main__.TestWebScraper.test_scrape_ticker_data_exists) ... ok
test_scrape_ticker_data_fills_DB (__main__.TestWebScraper.test_scrape_ticker_data_fills_DB) ... ok
test_webspider_class_exists (__main__.TestWebScraper.test_webspider_class_exists) ... ok

----------------------------------------------------------------------
Ran 3 tests in 5.508s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

## Test Data Cleaning

In [206]:
%sql sqlite:///:memory:

class TestStockDataCleaner(unittest.TestCase):

    def setUp(self):
        # Suppress stdout automatically
        self.patcher = patch('sys.stdout', new_callable=StringIO)
        self.mock_stdout = self.patcher.start()

    def tearDown(self):
        # Restore stdout after test
        self.patcher.stop()

    def test_simplify_company_names(self):
        test_data = {"A Group Inc.","B Group, Inc.", "C Holdings Inc.", "D Holdings, Inc.", "E Holdings Limited",
                     "F International Limited", "G Holdings LLC", "H International Limited", "I S.A.", "J Corp.",
                     "K Co.", "L Inc.", "M, Inc."}
        res = StockDataCleaner.simplify_company_names(test_data)
        self.assertSetEqual(res, {"A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M"})

suite = unittest.TestLoader().loadTestsFromTestCase(TestStockDataCleaner)
unittest.TextTestRunner(verbosity=2).run(suite)

test_simplify_company_names (__main__.TestStockDataCleaner.test_simplify_company_names) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.002s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

## Analysis

## Conclusions
- 

## Possible improvements for the project
1. Work on SQL-inject
2. SChedule with Pipeline
3. 

## Difficulties with data
scraper ahh a lot

## References
$^1$:

In [15]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
